# 05_00 Training on the Odyssey (overview)

*📖 Book: chapter 5*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** A new model (`HomerGreekLLM` in `src/homer_ai/model.py`) is full of random numbers, so its guesses are random (you saw it in [01_04](../01_foundations/01_04_our_model_as_a_black_box.ipynb)). Training turns them into Homer-like Greek.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What "training" means for a model, in plain words
- What the loss, a training step and an epoch are
- What the learning rate is, and why its size matters
- How to tell real learning from memorising
- Which notebook to open for each idea

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **training step** | one round of: guess → measure the loss → work out which way to nudge each weight → nudge them | `loss.backward()` then `optimizer.step()` |
| **epoch** | one full pass through all the training text | we train for 800 |
| **learning rate** | how big each nudge to the weights is | `lr=0.01` |
| **validation set** | text kept hidden from training, used only to test the model | the last 10% of the text |

Words you've met before: **weight** ([01_00](../01_foundations/01_00_chapter_overview.ipynb)), **loss** ([01_03](../01_foundations/01_03_how_a_program_learns.ipynb)), **optimizer** ([01_03](../01_foundations/01_03_how_a_program_learns.ipynb)), **chunk** ([02_04](../02_text_to_numbers/02_04_sliding_window.ipynb)).

## 3. The idea

### ❓ How does a model get better?

Training is revising for an exam with a pile of flashcards. The model guesses the back of a card, we turn the card over, and it adjusts itself a little. Then the next card, thousands of times. You know the catch from school: go through the same pile often enough and you know the cards by heart without having understood the subject. So we lock a few cards in a drawer where the model never sees them, and those are the exam. Where the picture is wrong: a student gets the idea after a handful of cards and knows why an answer is right, and our model needs thousands of tiny nudges and only ever shifts numbers.

In the model, one round of this is called a **training step**. It has four parts:

1. **Guess** the next letter for every position in a short piece of text (a chunk).
2. **Measure** how wrong the guesses were. That mistake score is the loss.
3. **Work out** which way to nudge each adjustable number (each weight) so the loss goes down.
4. **Nudge** the weights a little. The part of the code that does this is the optimizer.

Here's one real flashcard from Book 1, `ἄνδρ → α`, before and after training:

```
flashcard:   ἄνδρ → α
untrained:   chance for α 0.0056   →  loss 5.18   (a wild guess among 137)
trained:     chance for α 0.1194   →  loss 2.125  (its top 3: ο 0.218, ε 0.201, α 0.119)
```

So training raised the right answer's chance about 20 times, and the loss fell. You'll compute these numbers yourself in [05_01](05_01_loss_and_epochs.ipynb).

### ❓ What do those steps look like in our code?

They're in `train()` in `src/homer_ai/train_homer.py`. One word first: the model's scores for each possible next character are called **logits**.

```python
logits = model(inputs)                   # 1. guess
loss = F.cross_entropy(logits, targets) # 2. measure
optimizer.zero_grad()                    # wipe the last chunk's nudges
loss.backward()                          # 3. work out the nudges
optimizer.step()                         # 4. nudge
```

`optimizer.zero_grad()` clears the nudges left over from the last chunk. Without it they'd pile up, like not clearing a calculator between sums.

One pass through all the chunks of the text is an **epoch**. We train for 800 epochs.

**Look at:** the ring from the top, clockwise. Each box shows its step and the line of code that does it. The orange box, the loss, is the number training tries to lower.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    EDGE,
    INK,
    MUTED,
    box,
    diagram_axes,
    ring,
    studio_figure,
    tag,
)

fig = studio_figure("The training step", "Four things, again and again", (9, 6.5))
ax = diagram_axes(fig)
CX, CY = 5.0, 3.5

boxes = [
    ("1. guess", "logits = model(inputs)"),
    ("2. measure the loss", "F.cross_entropy(logits, targets)"),
    ("3. work out the nudges", "loss.backward()"),
    ("4. nudge the weights", "optimizer.step()"),
]
centres = ring(
    ax, CX, CY, 3.0, 1.9, [""] * 4, [EDGE, ACCENT, EDGE, EDGE], box_w=3.4, box_h=1.1
)
for (x, y), (name, code_line) in zip(centres, boxes, strict=True):
    tag(ax, x, y + 0.2, name, INK, 11, weight="bold")
    tag(ax, x, y - 0.22, code_line, MUTED, 8, mono=True)

tag(ax, CX, CY + 0.35, "one round per chunk", INK, 10)
tag(ax, CX, CY - 0.05, "all the chunks = 1 epoch", ACCENT, 10)
tag(ax, CX, CY - 0.45, "ἄνδρ → α is one flashcard", MUTED, 9)
box(ax, CX, 0.4, 5.0, 0.5, "", MUTED, dashed=True)
tag(ax, CX, 0.4, "before step 3: optimizer.zero_grad() wipes old nudges", MUTED, 8.5)
plt.show()

### ❓ How big should each nudge be?

That's the **learning rate**. Too big, and the model overshoots and the loss jumps up. Too small, and it learns very slowly. You can also change the size during training, big first and small later. That plan is called a learning-rate schedule.

### ❓ If the loss is low, has the model learned?

Not necessarily. It may just have memorised the text. To find out, keep some text hidden from training and test on that. The hidden text is the **validation set**.

### ❓ Which notebook do I open for each idea?

| Notebook | Question it answers |
|---|---|
| [`05_01_loss_and_epochs.ipynb`](05_01_loss_and_epochs.ipynb) | What's the loss, and how do I read a loss curve? |
| [`05_02_learning_rate.ipynb`](05_02_learning_rate.ipynb) | Why did the loss jump up, and how big should the steps be? |
| [`05_03_memorising_vs_understanding.ipynb`](05_03_memorising_vs_understanding.ipynb) | Is a low loss really good? |
| [`05_04_running_fair_experiments.ipynb`](05_04_running_fair_experiments.ipynb) | How do I change something and know what the change did? |
| [`05_05_validation.ipynb`](05_05_validation.ipynb) | Did the model learn, or memorise? |
| [`05_06_saving_and_loading.ipynb`](05_06_saving_and_loading.ipynb) | How do I keep the trained model, and use it later? |
| [`05_07_train_the_gpt.ipynb`](05_07_train_the_gpt.ipynb) | How is the GPT of chapter 4 trained, and how do I train my own? |

## 4. Recap

- Training repeats one step: guess, measure the loss, work out the nudges, nudge the weights.
- One pass through all the text is an epoch. The learning rate is the size of each nudge.
- A low loss can mean memorising. Hidden text (the validation set) shows the truth.

**Next:** [05_01: Loss and epochs](05_01_loss_and_epochs.ipynb).